<a href="https://colab.research.google.com/github/codefromaryan/Internship-Assignment-Project/blob/main/PrimeTrade_Ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Primetrade.ai Data Science Assignment
Basic structure for analyzing sentiment vs trader performance
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# PHASE 1: LOAD AND EXPLORE DATA

# Load Fear & Greed Index
fgi_df = pd.read_csv('fear_greed_index.csv')
print("\n Fear and Greed Index")
print(f"  Shape: {fgi_df.shape}")
print(f"  Columns: {list(fgi_df.columns)}")
print(f"  Date range: {fgi_df['date'].min()} to {fgi_df['date'].max()}")
print(f"\n  Sample data:\n{fgi_df.head()}")

# Load Historical Trade Data
trades_df = pd.read_csv('historical_data.csv')
print("\n Historical Trade Data")
print(f"  Shape: {trades_df.shape}")
print(f"  Columns: {list(trades_df.columns)}")
print(f"\n  Sample data:\n{trades_df.head()}")

In [ ]:
#  DATA CLEANING

# Check for missing values
print("\nMissing values in Fear & Greed Index:")
print(fgi_df.isnull().sum())

print("\nMissing values in Trade Data:")
print(trades_df.isnull().sum())

# Convert dates to datetime
fgi_df['date'] = pd.to_datetime(fgi_df['date'])
trades_df['date_extracted'] = pd.to_datetime(trades_df['Timestamp IST'].str.split().str[0], format='%d-%m-%Y')

# Basic statistics
print("\n[STATISTICS] Fear & Greed Index")
print(fgi_df[['value']].describe())

print("\n[STATISTICS] Trade Data - Closed PnL")
print(trades_df[['Closed PnL']].describe())

print("\n[STATISTICS] Trade Counts")
print(f"  Total trades: {len(trades_df)}")
print(f"  Unique traders: {trades_df['Account'].nunique()}")
print(f"  Unique coins: {trades_df['Coin'].nunique()}")

In [ ]:
# PHASE 2: FEATURE ENGINEERING

# Merge sentiment data with trades
trades_merged = trades_df.merge(
    fgi_df[['date', 'value', 'classification']],
    left_on='date_extracted',
    right_on='date',
    how='left'
)
print(f"Merged data shape: {trades_merged.shape}")

# Calculate trader-level metrics
trader_stats = trades_df.groupby('Account').agg({
    'Closed PnL': ['sum', 'mean', 'count', 'std'],
    'Side': lambda x: (x == 'BUY').sum()  # Count of buy trades
}).round(4)

trader_stats.columns = ['Total_PnL', 'Avg_PnL', 'Num_Trades', 'PnL_Std', 'Buy_Count']
trader_stats['Win_Rate'] = (trades_df.groupby('Account')['Closed PnL'].apply(lambda x: (x > 0).sum()) /
                             trades_df.groupby('Account').size() * 100).round(2)
trader_stats = trader_stats.sort_values('Total_PnL', ascending=False)

print("\nTop 10 Traders by Profitability:")
print(trader_stats.head(10))

In [ ]:
# PHASE 3: ANALYSIS

# Sentiment vs Performance
print("\n[ANALYSIS] Profitability by Sentiment Category")
sentiment_analysis = trades_merged.groupby('classification')['Closed PnL'].agg([
    'count', 'sum', 'mean', 'std'
]).round(4)
sentiment_analysis.columns = ['Num_Trades', 'Total_PnL', 'Avg_PnL', 'Std_Dev']
print(sentiment_analysis)

# Buy vs Sell Analysis
print("\n[ANALYSIS] Buy vs Sell Profitability")
buy_sell_analysis = trades_df.groupby('Side')['Closed PnL'].agg([
    'count', 'sum', 'mean'
]).round(4)
buy_sell_analysis.columns = ['Num_Trades', 'Total_PnL', 'Avg_PnL']
print(buy_sell_analysis)

# Top coins by volume and profitability
print("\n[ANALYSIS] Top Coins by Trading Volume")
coin_analysis = trades_df.groupby('Coin').agg({
    'Account': 'count',  # Number of trades
    'Closed PnL': 'sum'
}).round(2)
coin_analysis.columns = ['Trades', 'Total_PnL']
coin_analysis = coin_analysis.sort_values('Trades', ascending=False)
print(coin_analysis.head(10))

In [ ]:
# PHASE 4: VISUALIZATIONS

fig, axes = plt.subplots(3, 2, figsize=(18, 14))
axes = axes.flatten()

# 1. Sentiment Over Time
axes[0].plot(fgi_df['date'], fgi_df['value'], linewidth=2, color='steelblue')
axes[0].fill_between(fgi_df['date'], fgi_df['value'], alpha=0.3, color='steelblue')
axes[0].axhline(y=50, color='red', linestyle='--', label='Neutral (50)', alpha=0.5)
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Fear & Greed Index Score')
axes[0].set_title('Market Sentiment Over Time', fontweight='bold')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# 2. PnL vs Sentiment
if 'value' in trades_merged.columns:
    colors = {'BUY': 'green', 'SELL': 'red'}
    for side in trades_merged['Side'].unique():
        mask = trades_merged['Side'] == side
        axes[1].scatter(
            trades_merged[mask]['value'],
            trades_merged[mask]['Closed PnL'],
            alpha=0.5,
            label=side,
            s=50,
            color=colors.get(side, 'blue')
        )
    axes[1].set_xlabel('Fear & Greed Index Score')
    axes[1].set_ylabel('Closed PnL (USD)')
    axes[1].set_title('Trade Profitability vs Sentiment', fontweight='bold')
    axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].set_visible(False)

# 3. Win Rate by Sentiment Category
if 'classification' in trades_merged.columns:
    win_rate_by_sentiment = trades_merged.groupby('classification').apply(
        lambda x: (x['Closed PnL'] > 0).sum() / len(x) * 100
    ).sort_values(ascending=False)

    axes[2].bar(
        range(len(win_rate_by_sentiment)),
        win_rate_by_sentiment.values,
        color=['green' if x > 50 else 'red' for x in win_rate_by_sentiment.values]
    )
    axes[2].set_xticks(range(len(win_rate_by_sentiment)))
    axes[2].set_xticklabels(win_rate_by_sentiment.index, rotation=45)
    axes[2].set_ylabel('Win Rate (%)')
    axes[2].set_title('Win Rate by Market Sentiment', fontweight='bold')
    axes[2].axhline(y=50, color='black', linestyle='--', alpha=0.5, label='50% Neutral')
    axes[2].legend()
else:
    axes[2].set_visible(False)

# 4. Top Traders
top_traders = trader_stats.head(10)
traders_short = [acc[:10] + '...' for acc in top_traders.index]
axes[3].barh(
    traders_short,
    top_traders['Total_PnL'],
    color=['green' if x > 0 else 'red' for x in top_traders['Total_PnL']]
)
axes[3].set_xlabel('Total PnL (USD)')
axes[3].set_title('Top 10 Traders by Profitability', fontweight='bold')

# 5. Buy vs Sell Performance
sides = buy_sell_analysis.index
values = buy_sell_analysis['Avg_PnL']
colors_bs = ['green' if x > 0 else 'red' for x in values]
axes[4].bar(sides, values, color=colors_bs, alpha=0.7, edgecolor='black')
axes[4].set_ylabel('Average PnL per Trade (USD)')
axes[4].set_title('Average Profitability: Buy vs Sell', fontweight='bold')
axes[4].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
for i, v in enumerate(values):
    axes[4].text(i, v + (2 if v > 0 else -2), f'${v:.2f}', ha='center', fontweight='bold')

# Hide empty 6th subplot
axes[5].set_visible(False)

plt.tight_layout()
plt.show()



In [ ]:
# PHASE 5: KEY INSIGHTS

print("\n FINDING 1: Sentiment Impact")
best_sentiment = sentiment_analysis['Avg_PnL'].idxmax()
worst_sentiment = sentiment_analysis['Avg_PnL'].idxmin()
print(f"  • Best performing sentiment: {best_sentiment} (Avg PnL: ${sentiment_analysis.loc[best_sentiment, 'Avg_PnL']:.2f})")
print(f"  • Worst performing sentiment: {worst_sentiment} (Avg PnL: ${sentiment_analysis.loc[worst_sentiment, 'Avg_PnL']:.2f})")

print("\n FINDING 2: Trading Strategy")
best_side = buy_sell_analysis['Avg_PnL'].idxmax()
print(f"  • More profitable strategy: {best_side}")
print(f"  • Average PnL: ${buy_sell_analysis.loc[best_side, 'Avg_PnL']:.2f}")

print("\n FINDING 3: Top Performer")
top_trader = trader_stats.index[0]
print(f"  • Account: {top_trader}")
print(f"  • Total PnL: ${trader_stats.iloc[0]['Total_PnL']:.2f}")
print(f"  • Win Rate: {trader_stats.iloc[0]['Win_Rate']:.2f}%")
print(f"  • Trades: {int(trader_stats.iloc[0]['Num_Trades'])}")
